<a href="https://colab.research.google.com/github/hanjeongseop/Data/blob/main/4_2_PDA(%EC%98%88%EC%B8%A1%EC%A0%81_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%B6%84%EC%84%9D).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IV. PDA (예측적 데이터 분석)

## 1. 분류 모델을 이용한 불량 분류 모델 구성하기

In [ ]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy.stats as stats

In [ ]:
# 파일 읽기 및 내용 확인하기

github_csv_url = 'https://raw.githubusercontent.com/hanjeongseop/Data/main'

df1 = pd.read_csv(github_csv_url + '/preprocessing_data.csv')

df1

,Unnamed: 0,Ox_Chamber,process,type,Temp_OXid,Vapor,ppm,Pressure,Oxid_time,thickness,...,Flux480s,Flux840s,input_Energy,Temp_implantation,Furance_Temp,RTA_Temp,Target,Error_message,target_binom,Chamber_Route
0,0,1,Oxidation,dry,1138.979159,O2,32.80,0.200,62.0,699.443,...,3.002593e+17,6.000007e+17,31574.410,102.847,885.0,154,96,none,0.0,route_11133
1,1,1,Oxidation,dry,1218.184551,O2,31.86,0.194,137.0,696.792,...,3.017903e+17,6.000012e+17,31580.213,104.323,919.0,154,102,none,0.0,route_11222
2,2,1,Oxidation,dry,1062.467808,O2,39.51,0.217,128.0,705.471,...,2.994231e+17,6.000002e+17,32162.414,100.605,916.0,155,95,none,0.0,route_11311
3,3,1,Oxidation,dry,1114.704773,O2,32.88,0.201,90.0,710.772,...,2.991354e+17,6.000003e+17,32874.925,101.739,911.0,156,117,none,0.0,route_12111
4,4,1,Oxidation,dry,989.411946,O2,38.11,0.204,98.0,716.975,...,3.005576e+17,6.000013e+17,30985.928,106.422,872.0,155,143,none,0.0,route_12222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,845,3,Oxidation,wet,1280.687973,H2O,45.19,0.214,21.0,708.586,...,2.995317e+17,5.999986e+17,32252.961,101.177,868.0,152,84,none,0.0,route_33111
748,846,3,Oxidation,wet,1275.153349,H2O,45.08,0.215,22.0,712.936,...,3.004926e+17,5.999991e+17,32253.818,100.736,868.0,151,105,none,0.0,route_33222
749,847,3,Oxidation,wet,1275.182502,H2O,45.10,0.214,21.0,715.498,...,3.009325e+17,6.000003e+17,32248.621,101.503,868.0,152,78,none,0.0,route_33333
750,848,1,Oxidation,wet,1268.105427,H2O,45.07,0.215,22.0,707.179,...,3.006733e+17,6.000003e+17,32241.426,101.061,867.0,152,42,none,0.0,route_11133


In [ ]:
# X 데이터 와 Y 데이터 설정
# target_binom => 불량예측

df1['target_binom'].value_counts()

,count
target_binom,
0.0,688
1.0,64


In [ ]:
df1.columns

Index(['Unnamed: 0', 'Ox_Chamber', 'process', 'type', 'Temp_OXid', 'Vapor',
       'ppm', 'Pressure', 'Oxid_time', 'thickness', 'No_Die', 'Reinforcement',
       'Unnamed: 0_x', 'photo_soft_Chamber', 'process 2', 'resist_target',
       'N2_HMDS', 'pressure_HMDS', 'temp_HMDS', 'temp_HMDS_bake',
       'time_HMDS_bake', 'spin1', 'spin2', 'spin3', 'photoresist_bake',
       'temp_softbake', 'time_softbake', 'lithography_Chamber', 'Line_CD',
       'UV_type', 'Wavelength', 'Resolution', 'Energy_Exposure', 'Range_check',
       'Unnamed: 0_y', 'Etching_Chamber', 'Process 3', 'Temp_Etching',
       'Source_Power', 'Selectivity', 'Thin Film 4', 'Thin Film 3',
       'Thin Film 2', 'Thin Film 1', 'Etching_rate', 'Chamber_Num', 'process4',
       'Flux60s', 'Flux90s', 'Flux160s', 'Flux480s', 'Flux840s',
       'input_Energy', 'Temp_implantation', 'Furance_Temp', 'RTA_Temp',
       'Target', 'Error_message', 'target_binom', 'Chamber_Route'],
      dtype='object')

In [ ]:
# 공정단계에서 핵심 변수들을 추출

Y = df1['target_binom']
X = df1[['Oxid_time', 'thickness', 'resist_target', 'Line_CD', 'Etching_rate', 'input_Energy', 'Temp_implantation']]

In [ ]:
X

,Oxid_time,thickness,resist_target,Line_CD,Etching_rate,input_Energy,Temp_implantation
0,62.0,699.443,1.211940,30.959,2.75950,31574.410,102.847
1,137.0,696.792,0.887720,29.653,2.72775,31580.213,104.323
2,128.0,705.471,1.113156,28.063,2.67000,32162.414,100.605
3,90.0,710.772,0.882195,31.556,2.74825,32874.925,101.739
4,98.0,716.975,0.834001,31.969,2.74625,30985.928,106.422
...,...,...,...,...,...,...,...
747,21.0,708.586,0.923802,35.404,2.67450,32252.961,101.177
748,22.0,712.936,0.837348,31.011,2.72725,32253.818,100.736
749,21.0,715.498,0.859869,32.525,2.72275,32248.621,101.503
750,22.0,707.179,0.914315,28.001,2.69150,32241.426,101.061


In [ ]:
# 머신러닝을 위한 환경 세팅

# 학습 데이터와 검증 데이터를 분할
from sklearn.model_selection import train_test_split

# 알고리즘 : 컴퓨터가 데이터에서 규칙을 찾아낼 수 있도록 도움을 주는 수학 구조
# 알고리즘을 적용
from sklearn.tree import DecisionTreeClassifier

# 평가 지표 : 학습이 얼마나 잘 되었는가? / 일반화가 얼마나 잘 되었는가?를 판단
from sklearn.metrics import accuracy_score

In [ ]:
# 학습 데이터 70%와 검증 데이터 30% 분할

train_test_split(X,Y, test_size=0.3)

[     Oxid_time  thickness  resist_target  Line_CD  Etching_rate  input_Energy  \
 410      103.0    716.015       1.072506   57.004       2.71650     31179.148   
 414       56.0    703.906       0.906802   28.301       2.67225     32007.126   
 389      152.0    704.964       0.929738   36.495       2.71825     31254.326   
 310       94.0    708.346       1.028761   66.532       2.73775     32773.392   
 153      106.0    705.357       1.089758   35.574       2.72850     31485.171   
 ..         ...        ...            ...      ...           ...           ...   
 594      230.0    719.948       0.891425   49.549       2.71250     32469.850   
 562      222.0    711.408       0.871088   50.833       2.70700     32300.976   
 185      138.0    708.001       0.960653   44.480       2.73050     32069.305   
 736       26.0    717.937       0.862463   48.896       2.72750     32257.524   
 39        55.0    705.424       1.166580   30.188       2.69825     31818.475   
 
      Temp_imp

In [ ]:
# 학습 데이터 70%와 검증 데이터 30% 분할

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3)

In [ ]:
# 학습 데이터로 데이터간 규칙을 도출하는 절차  => 모델
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

DecisionTreeClassifier()

In [ ]:
# 평가 : 모델에 의해서 예측된 예측값과 실제값과 비교하는 절차
y_train_pred = model.predict(X_train)   # 예측값

In [ ]:
# 실제값과 예측값을 비교해 성능을 평가

accuracy_score(y_train, y_train_pred)

# 학습 성능을 확인

1.0

In [ ]:
# 검증 데이터의 예측값 계산
# 데이터를 동적으로 70:30 비율로 구분했기때문에 작업 컴퓨터에 따라 결과가 달라질 수 있음

y_test_pred = model.predict(X_test)
accuracy_score(y_test, y_test_pred)

# 검증 성능을 확인

0.8938053097345132

In [ ]:
## 데이터 예측 적용
## 예측값 : Y = df1['target_binom']
## 변수 : X = df1[['Oxid_time', 'thickness', 'resist_target', 'Line_CD', 'Etching_rate', 'input_Energy', 'Temp_implantation']]

# 새로운 값을 외부로 부터 입력받아 Wafer의 불량 여부를 분류해보자.
input_x1 = input('Oxid_time 값을 입력하시오. : ')
input_x2 = input('thickness 값을 입력하시오. : ')
input_x3 = input('resist_target 값을 입력하시오. : ')
input_x4 = input('Line_CD 값을 입력하시오. : ')
input_x5 = input('Etching_rate 값을 입력하시오. : ')
input_x6 = input('input_Energy 값을 입력하시오. : ')
input_x7 = input('Temp_implantation 값을 입력하시오. : ')

Oxid_time 값을 입력하시오. : 1
thickness 값을 입력하시오. : 2
resist_target 값을 입력하시오. : 3
Line_CD 값을 입력하시오. : 4
Etching_rate 값을 입력하시오. : 5
input_Energy 값을 입력하시오. : 6
Temp_implantation 값을 입력하시오. : 7


In [ ]:
input_data = pd.DataFrame([[input_x1,input_x2,input_x3,input_x4,input_x5,input_x6,input_x7]])

In [ ]:
model.predict(input_data)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


array([0.])

In [ ]:
# model.predict(input_data)를 수행한 결과가 0으로 예측됨  => 입력값을 조정해 다른 결과도 확인해보자
# 입력된 데이터를 기준으로 예측한 결과 문제가 없다,

df1['target_binom'].value_counts()

,count
target_binom,
0.0,688
1.0,64


## 2. 특성 공학을 이용해 성능 높이기

## 전체 데이터가 1000 중 정상 900 / 불량 100 일때 학습 데이터와 검증 데이터를 70:30으로 구분

### 학습데이터 : 정상 650 / 불량 50
### 검증데이터 : 정상 250 / 불량 50

### 이런 경우 정확도(accuracy score)만 가지고 판단하면 안된다.

## 이를 위해 정밀도(Precision)와 재현율(Recall) 이라는 지표 확인이 필요하다.

### 정밀도 : 예측을 불량으로 한 데이터 중, 불량을 불량으로 잘 맞춘 데이터의 비율
### 재현율 : 실제 불량인 데이터 중, 불량을 불량으로 잘 맞춘 데이터의 비율
### F1 Score : 정밀도와 재현율을 비율로 표현 [ (2 * 정밀도 * 재현율) / (정밀도 + 재현율) ]

In [ ]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy.stats as stats

In [ ]:
df1 = pd.read_csv(github_csv_url + '/modeling_data.csv')

df1.head()

,Unnamed: 0,Oxid_time,thickness,resist_target,Line_CD,Etching_rate,input_Energy,Temp_implantation,target_binom
0,0,62.0,699.443,1.211940,30.959,2.75950,31574.410,102.847,0.0
1,1,137.0,696.792,0.887720,29.653,2.72775,31580.213,104.323,0.0
2,2,128.0,705.471,1.113156,28.063,2.67000,32162.414,100.605,0.0
3,3,90.0,710.772,0.882195,31.556,2.74825,32874.925,101.739,0.0
4,4,98.0,716.975,0.834001,31.969,2.74625,30985.928,106.422,0.0


In [ ]:
# 분석에 불필요한 컬럼을 제거해보자
# 의미없는 항목 Unnamed: 0 제거
# 목표 변수로 지정했기 때문에 제거 target_binom

Y = df1['target_binom']
X = df1.drop(columns=['Unnamed: 0', 'target_binom'])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# 분류모델에서 모든 평가 지표를 확인할 수 있다.
from sklearn.metrics import classification_report

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=1234)

In [ ]:
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

DecisionTreeClassifier()

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

In [ ]:
print(classification_report(y_train, y_train_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       482
         1.0       1.00      1.00      1.00        44

    accuracy                           1.00       526
   macro avg       1.00      1.00      1.00       526
weighted avg       1.00      1.00      1.00       526



In [ ]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

         0.0       0.94      0.95      0.94       206
         1.0       0.41      0.35      0.38        20

    accuracy                           0.90       226
   macro avg       0.67      0.65      0.66       226
weighted avg       0.89      0.90      0.89       226



## 학습데이터의 성능은 100%인 반면 검증데이터의 성능 중 불량데이터이 경우 40%의 성능을 나타내고 있다.

### 이런한 경우를 과적합(Overfitting)이라고 한다.
### 학습을 너무 자세하게 진행해서 학습데이터에 최적화된 상태로 새로운 데이터가 들어왔을때 매우 저조한 결과가 나타나는 현상

In [ ]:
# 1. F1 Score가 저조하다. 이는 데이터의 비율이 깨져있다.(정상 데이터에 비해 불량 비율이 10% = > 70:30 비율에 어긋남)

Y.value_counts()

,count
target_binom,
0.0,688
1.0,64


### #################################
## 해결해야 하는 문제
### #################################
## 1. 과적합
## 2. F1 Score가 저조하다. 이는 데이터의 비율이 깨져있다는 것이다.


### #################################
## 해결 방법
### #################################
## 1. 특성공학 : 학습이 잘 수행됨과 동시에, 일반화 성능도 높이기 위해 데이터를 정제하는 작업
## 2. 알고리즘 : 여러 종류의 알고리즘을 비교하여, 가장 성능이 높은 알고리즘을 찾는다.

# 특성공학
## 1.1 Impute : 결측치를 적절한 값으로 처리
## 1.2 Scaling : 숫자형 자료의 스케일을 맞춰주는 작업 (선형성을 이용한 머신러닝 알고리즘)
## 1.3 Cross Validation : 검증 데이터를 교차로 변경해가며, 여러 가지 모델을 동시에 학습
## 1.4  Hyper Parameter Tuning :  알고리즘 내 수학적 구조를 사용자가 임의로 통제하는 방식
## 1.5  Imbalanced Data Sampling : 데이터의 비율이 깨져있는 경우, 데이터의 비율을 임의로 맞추는 작업

In [ ]:
# sklearn 내에서 Pipeline 구조를 이용해, 특성공학과 학습이 동시에 진행되도록 구축

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

# Scaling 추가
from sklearn.preprocessing import MinMaxScaler # 최소값 0 / 최대값 1

In [ ]:
# 2. 스케일링 진행 및 2.학습 수행

pipe_list = [('scaler', MinMaxScaler()), ('model', DecisionTreeClassifier())]
pipe_model = Pipeline(pipe_list)

In [ ]:
# 3. 교차검증 작업과 매개변수(Hyper parameter)튜닝을 동시에 수행하는 GridsearchCV
# 교차검증은 3회 진행,

grid_model = GridSearchCV(pipe_model, cv = 3, param_grid= {})

In [ ]:
# 4. 학습데이터를 넣어 수식을 생성
grid_model.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('scaler', MinMaxScaler()),
                                       ('model', DecisionTreeClassifier())]),
             param_grid={})

In [ ]:
# 가장 성능이 좋은 모델 도출

grid_model.best_estimator_

Pipeline(steps=[('scaler', MinMaxScaler()),
                ('model', DecisionTreeClassifier())])

In [ ]:
# 가장 성능이 좋은 모델 선택
best_model = grid_model.best_estimator_

In [ ]:
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

In [ ]:
print(classification_report(y_train, y_train_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       482
         1.0       1.00      1.00      1.00        44

    accuracy                           1.00       526
   macro avg       1.00      1.00      1.00       526
weighted avg       1.00      1.00      1.00       526



In [ ]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

         0.0       0.94      0.95      0.94       206
         1.0       0.41      0.35      0.38        20

    accuracy                           0.90       226
   macro avg       0.67      0.65      0.66       226
weighted avg       0.89      0.90      0.89       226



### [해석]

### 결과를 보니 과적합이 존재하고 검증 성능은 차이가 없어보인다.

### 이런경우
###  - 과적합을 해결하기 위해 매개변수(Hyper parameter)튜닝을 점검해봐야 한다.
###  - 교차검증 작업과 매개변수 튜닝을 동시에 진행할 때 가장 좋은 모델을 선택해야 한다.
###    위에서는 정확도가 높은 기준을 적용해 모델을 선정했기 때문에 정확도가 100%이다
###    만일, F1 SCORE가 높은 모델을 선택하고 싶다라는 가정을 세우면
###    grid_model = GridSearchCV(pipe_model, cv = 3, param_grid= {}, scoring='f1')으로 변경하면 된다.

In [ ]:
# 3. 교차검증 작업과 매개변수(Hyper parameter)튜닝을 동시에 수행하는 GridsearchCV

grid_model = GridSearchCV(pipe_model, cv = 3, param_grid= {}, scoring='f1')

In [ ]:
# 4. 학습데이터를 넣어 수식을 생성
grid_model.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('scaler', MinMaxScaler()),
                                       ('model', DecisionTreeClassifier())]),
             param_grid={}, scoring='f1')

In [ ]:
# 가장 성능이 좋은 모델 선택
best_model = grid_model.best_estimator_

In [ ]:
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

In [ ]:
print(classification_report(y_train, y_train_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       482
         1.0       1.00      1.00      1.00        44

    accuracy                           1.00       526
   macro avg       1.00      1.00      1.00       526
weighted avg       1.00      1.00      1.00       526



In [ ]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

         0.0       0.94      0.93      0.93       206
         1.0       0.33      0.35      0.34        20

    accuracy                           0.88       226
   macro avg       0.63      0.64      0.64       226
weighted avg       0.88      0.88      0.88       226



In [ ]:
# 과적합을 해결해보자

# 3. 교차검증 작업과 매개변수(Hyper parameter)튜닝을 동시에 수행하는 GridsearchCV
# 의사결정나무의 파라미터 설정값을 세팅해보자.

grid_model = GridSearchCV(pipe_model, cv = 3,
                          param_grid= {'model__max_depth':range(5,10),
                                      'model__min_samples_leaf':range(5,10),
                                      'model__class_weight':['balanced',None]},
                          scoring='f1')

In [ ]:
# 4. 학습데이터를 넣어 수식을 생성
grid_model.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('scaler', MinMaxScaler()),
                                       ('model', DecisionTreeClassifier())]),
             param_grid={'model__class_weight': ['balanced', None],
                         'model__max_depth': range(5, 10),
                         'model__min_samples_leaf': range(5, 10)},
             scoring='f1')

In [ ]:
# 가장 성능이 좋은 모델 선택
best_model = grid_model.best_estimator_

In [ ]:
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

In [ ]:
print(classification_report(y_train, y_train_pred))

              precision    recall  f1-score   support

         0.0       0.95      0.99      0.97       482
         1.0       0.74      0.45      0.56        44

    accuracy                           0.94       526
   macro avg       0.85      0.72      0.77       526
weighted avg       0.93      0.94      0.93       526



In [ ]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

         0.0       0.94      0.97      0.95       206
         1.0       0.54      0.35      0.42        20

    accuracy                           0.92       226
   macro avg       0.74      0.66      0.69       226
weighted avg       0.90      0.92      0.91       226



## *** 실제 실무에서 가장 많이 사용하는 일반적인 방법입니다.